# 09 · Vizualizace perceptronu

Tento notebook je **edukační částí** cvičení 09. Prochází přednášku o jednom neuronu: perceptronové pravidlo,
lineární a logistický neuron a metriky klasifikace a regrese. Každá metoda běží na malých syntetických datech
a má graf s **ovládacími prvky** (posuvníky, nabídky, zaškrtávací pole). Kód nemusíte číst ani měnit.

**Jak pracovat:** spusťte všechny buňky (*Run → Run All Cells*). U každé části se nad grafem objeví ovládání; každá změna graf
překreslí. Kódové buňky jsou sbalené; pokud vás kód zajímá, rozbalíte ho kliknutím na lištu vlevo od buňky (nebo na tři tečky).
Každá část končí **✏️ Úkolem**; odpovědi vám pomohou při ladění v praktické části (`perceptron-practice-cz.ipynb`).


Pokud se ovládání nezobrazí, chybí balíček `ipywidgets`
(viz README).

In [ ]:
import numpy as np                          # pole a výpočty
import pandas as pd                         # tabulky výsledků
import matplotlib.pyplot as plt             # grafy
import ipywidgets as widgets                # posuvníky, nabídky a zaškrtávací pole
from IPython.display import display         # přehledné tabulky
from scipy.stats import norm                # normální rozdělení (teoretické AUC, ideální pravděpodobnosti)

BLUE, ORANGE, TEAL, GREY = "#0065BD", "#C2410C", "#00897B", "#8C96A0"  # barvy jako v přednášce
plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True, "grid.alpha": 0.3,  # jednotný vzhled grafů
                     "axes.spines.top": False, "axes.spines.right": False})
SEED = 0                                    # semínko: všichni dostanou stejná data
STYLE = {"description_width": "190px"}      # místo pro popisky ovládání
WIDE = widgets.Layout(width="95%", max_width="560px")  # v úzkém okně se zúží


def fslider(label, value, lo, hi, step, fmt=".2f", live=False):
    # posuvník pro desetinné číslo; live=True překresluje během tažení
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=label, readout_format=fmt,
                               continuous_update=live, style=STYLE, layout=WIDE)


def islider(label, value, lo, hi, step=1, live=False):
    # posuvník pro celé číslo
    return widgets.IntSlider(value=value, min=lo, max=hi, step=step, description=label,
                             continuous_update=live, style=STYLE, layout=WIDE)


def choice(label, options, value):
    # posuvník přes několik daných hodnot
    return widgets.SelectionSlider(options=options, value=value, description=label,
                                   continuous_update=False, style=STYLE, layout=WIDE)


def menu(label, options, value):
    # rozbalovací nabídka; možnosti jsou dvojice (text, hodnota)
    return widgets.Dropdown(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def controls(draw, **inputs):
    # zobrazí ovládání a po každé změně překreslí graf
    widgets.interact(draw, **inputs)


def rng(seed=SEED):
    # nový generátor s pevným semínkem -> po každé změně stejná data
    return np.random.default_rng(seed)


def perceptron(X, y, w0=(0.0, 0.0), b0=0.0, eta=1.0, order=None, epochs=100, g=None, bias=True):
    # perceptronové pravidlo z přednášky: y ∈ {−1, +1}, vzorek je chybou, když y·s ≤ 0
    w, b = np.array(w0, dtype=float), float(b0)
    history = [(w.copy(), b)]               # váhy po každé opravě
    errors_per_epoch = []
    for epoch in range(epochs):
        if g is not None:
            idx = g.permutation(len(y))     # náhodné pořadí v každé epoše
        else:
            idx = order if order is not None else range(len(y))  # pevné pořadí
        errors = 0
        for i in idx:
            s = w @ X[i] + b                # skóre vzorku
            if y[i] * s <= 0:               # špatná strana přímky nebo přímo na ní
                w = w + eta * y[i] * X[i]   # posun přímky ke vzorku
                if bias:
                    b = b + eta * y[i]
                errors += 1
                history.append((w.copy(), b))
        errors_per_epoch.append(errors)
        if errors == 0:                     # celá epocha bez chyby: konec
            break
    return w, b, errors_per_epoch, history


def count_errors(X, y, w, b):
    # počet trénovacích vzorků na špatné straně (nebo na přímce)
    return int(np.sum(y * (X @ w + b) <= 0))


def draw_line(ax, w, b, xlim, **style):
    # nakreslí přímku w₁x₁ + w₂x₂ + b = 0
    x = np.array(xlim, dtype=float)
    if abs(w[1]) > 1e-12:
        ax.plot(x, -(w[0] * x + b) / w[1], **style)
    elif abs(w[0]) > 1e-12:
        ax.axvline(-b / w[0], **style)      # svislá přímka, když w₂ = 0


def show_classes(ax, X, y, s=30, alpha=0.9):
    # třída +1 (nebo 1) jako oranžové kosočtverce, třída −1 (nebo 0) jako modrá kolečka
    pos = y > 0
    ax.scatter(*X[~pos].T, s=s, color=BLUE, alpha=alpha, label="třída −1" if y.min() < 0 else "třída 0")
    ax.scatter(*X[pos].T, s=s * 1.2, color=ORANGE, marker="D", alpha=alpha, label="třída +1" if y.min() < 0 else "třída 1")


def two_clouds(g, n, distance, spread=1.0, gap=None):
    # dva gaussovské mraky na úhlopříčce, jejichž středy jsou od sebe `distance`
    c = np.array([1, 1]) / np.sqrt(2) * distance / 2
    X = np.r_[g.normal(c, spread, (n, 2)), g.normal(-c, spread, (n, 2))]
    y = np.r_[np.ones(n), -np.ones(n)].astype(int)
    if gap is not None:                     # odstranit body z pásu kolem x₁ + x₂ = 0 -> oddělitelná data
        keep = y * (X @ np.array([1, 1])) / np.sqrt(2) > gap / 2
        X, y = X[keep], y[keep]
    return X, y


print("Připraveno. Spusťte další buňky; pod každou částí se objeví ovládání.")

## 1. Perceptronové pravidlo krok za krokem

**Cíl metody:** najít váhy **w** a práh b tak, aby přímka w₁x₁ + w₂x₂ + b = 0 oddělila dvě třídy.
Skóre vzorku je s = **w**·**x** + b, předpověď je znaménko s. Vzorky se procházejí postupně; když je vzorek
zařazen chybně (y·s ≤ 0), přímka se posune k němu:

**w** ← **w** + η·y·**x**, b ← b + η·y.

Data jsou čtyři body z přednášky: x₁ = (3, 2) a x₃ = (1, 3) ve třídě +1, x₂ = (0, −2) a x₄ = (−1, 2) ve třídě −1.
S η = 1, nulovým startem a pořadím x₁, x₂, x₃, x₄ je výsledek z přednášky **w** = (4, 2), b = −1 po 3 opravách.

**Ovládání:** *krok* prochází vyhodnocení jedno po druhém (zakroužkovaný bod je právě vyhodnocený vzorek, čárkovaná přímka
předchozí hranice); ostatní prvky mění rychlost učení, start a pořadí. Tabulka ukazuje vyhodnocení až po aktuální krok.

In [ ]:
X1 = np.array([[3, 2], [0, -2], [1, 3], [-1, 2]], dtype=float)  # čtyři body z přednášky
y1 = np.array([1, -1, 1, -1])                                    # jejich třídy


def section1(step, eta, start, order):
    (w0, b0) = start
    w, b = np.array(w0, dtype=float), float(b0)
    log = []                                     # jeden záznam na vyhodnocení
    for epoch in range(1, 21):
        errors = 0
        for i in order:
            s = w @ X1[i] + b
            before = (w.copy(), b)
            wrong = y1[i] * s <= 0
            if wrong:
                w, b = w + eta * y1[i] * X1[i], b + eta * y1[i]
                errors += 1
            log.append({"epocha": epoch, "vzorek": f"x{i + 1}", "y": y1[i], "s": s,
                        "oprava": "ano" if wrong else "", "w₁": w[0], "w₂": w[1], "b": b,
                        "_i": i, "_before": before, "_after": (w.copy(), b)})
        if errors == 0:
            break
    k = min(step, len(log))                      # po konvergenci se nic nemění
    w_k, b_k = log[k - 1]["_after"] if k else (np.array(w0, dtype=float), float(b0))

    fig, ax = plt.subplots(figsize=(6, 6))
    for rec in log[:k]:                          # dřívější hranice, slabě
        draw_line(ax, *rec["_after"], (-3, 5), color=TEAL, alpha=0.15, lw=1)
    if k:
        draw_line(ax, *log[k - 1]["_before"], (-3, 5), color=GREY, ls="--", lw=1.5, label="před tímto krokem")
        cur = X1[log[k - 1]["_i"]]
        ax.scatter(*cur, s=600, facecolors="none", edgecolors="black", lw=2)   # právě vyhodnocený vzorek
    draw_line(ax, w_k, b_k, (-3, 5), color=TEAL, lw=3, label="aktuální hranice")
    show_classes(ax, X1, y1, s=90)
    for i, (xa, xb) in enumerate(X1):
        ax.annotate(f"x{i + 1}", (xa, xb), textcoords="offset points", xytext=(8, 6), fontsize=12)
    ax.set(xlim=(-3, 5), ylim=(-4, 5), xlabel="x₁", ylabel="x₂",
           title=f"krok {k}/{len(log)}: w = ({w_k[0]:g}, {w_k[1]:g}), b = {b_k:g}")
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=9)
    plt.show()
    table = pd.DataFrame(log[:k]).drop(columns=["_i", "_before", "_after"]) if k else pd.DataFrame()
    display(table)
    updates = sum(r["oprava"] != "" for r in log)
    w_end, b_end = log[-1]["_after"]
    print(f"na konci: w = ({w_end[0]:g}, {w_end[1]:g}), b = {b_end:g} po "
          f"{updates} opravách, nejmenší vzdálenost bodu od přímky "
          f"{(np.abs(X1 @ w_end + b_end) / np.linalg.norm(w_end)).min():.2f}")
    if (eta, start, order) == (1.0, ((0.0, 0.0), 0.0), (0, 1, 2, 3)):
        assert np.allclose(w_end, [4, 2]) and b_end == -1 and updates == 3   # výsledek z přednášky


controls(section1,
         step=islider("krok (vyhodnocení)", 4, 0, 16, live=True),
         eta=choice("rychlost učení η", [0.25, 0.5, 1.0, 2.0], 1.0),
         start=menu("start w, b", [("w = (0, 0), b = 0", ((0.0, 0.0), 0.0)), ("w = (−1, 1), b = 0", ((-1.0, 1.0), 0.0)),
                                                 ("w = (1, −1), b = 1", ((1.0, -1.0), 1.0))], ((0.0, 0.0), 0.0)),
         order=menu("pořadí v epoše", [("x₁, x₂, x₃, x₄", (0, 1, 2, 3)), ("x₂, x₁, x₃, x₄", (1, 0, 2, 3)),
                                                          ("x₃, x₁, x₂, x₄", (2, 0, 1, 3)), ("x₄, x₃, x₂, x₁", (3, 2, 1, 0))], (0, 1, 2, 3)))

**✏️ Úkol 1**

* Projděte první dvě epochy (kroky 1–8) a porovnejte s přednáškou. Proč se x₂ v kroku 6 počítá jako chyba, ačkoli jeho skóre není kladné?
* Zvolte pořadí x₂, x₁, x₃, x₄ a posuňte krok na konec. Jakou přímku dostanete? Je to také správné řešení? Která z obou přímek má větší nejmenší vzdálenost?
* Nastavte η = 0,5 s nulovým startem. Jak se změní váhy a jak přímka? Proč? Pak zvolte start w = (−1, 1): záleží teď na η?

## 2. Konvergence a mez počtu oprav

**Cíl metody:** vědět, kdy se perceptron zaručeně zastaví. Pokud nějaká přímka odděluje třídy s **okrajem** γ
(žádný bod k ní není blíž než γ) a všechny body leží v poloměru R, udělá perceptron nejvýše **(R/γ)²** oprav
(Novikoffova věta). Mez nezávisí na počtu bodů ani na jejich pořadí.

Body leží v kruhu o poloměru 3 a odděluje je přímka x₁ + x₂ = 0 s prázdným pásem o poloviční šířce γ.
Přímka prochází počátkem, perceptron se proto učí bez prahu. Deset běhů, každý s novými daty a náhodným pořadím.

**Ovládání:** okraj γ.

In [ ]:
def disc_data(g, gamma, n=200, radius=3.0):
    # body v kruhu, bez pásu |d| ≤ γ kolem přímky x₁ + x₂ = 0
    P = g.uniform(-radius, radius, (n * 4, 2))
    P = P[np.linalg.norm(P, axis=1) <= radius]
    d = P @ np.array([1, 1]) / np.sqrt(2)             # znaménková vzdálenost od přímky
    keep = np.abs(d) > gamma
    P, d = P[keep][:n], d[keep][:n]
    return P, np.where(d > 0, 1, -1)


def section2(gamma):
    updates, bounds = [], []
    for run in range(10):
        X, y = disc_data(rng(100 + run), gamma)
        w, b, errs, hist = perceptron(X, y, g=rng(run), bias=False, epochs=5000)
        R = np.linalg.norm(X, axis=1).max()                         # poloměr dat
        margin = np.min(np.abs(X @ np.array([1, 1])) / np.sqrt(2))  # skutečný okraj pravé přímky
        updates.append(len(hist) - 1)
        bounds.append((R / margin) ** 2)
    fig, axs = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 1.4]})
    t = np.array([-3, 3])
    axs[0].fill_between(t, -t - gamma * np.sqrt(2), -t + gamma * np.sqrt(2), color=GREY, alpha=0.25, label="pás ±γ")
    show_classes(axs[0], X, y, s=12)
    draw_line(axs[0], w, 0.0, (-3, 3), color=TEAL, lw=2, label="perceptron (poslední běh)")
    axs[0].set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), xlabel="x₁", ylabel="x₂")
    axs[0].set_aspect("equal")
    axs[0].legend(loc="lower left", fontsize=8)
    axs[1].bar(range(1, 11), updates, color=BLUE, label="opravy do konvergence")
    axs[1].axhline(np.mean(bounds), color=ORANGE, ls="--", label=f"mez (R/γ)² ≈ {np.mean(bounds):.0f}")
    axs[1].set(xlabel="běh", ylabel="počet oprav", yscale="log", xticks=range(1, 11))
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"γ = {gamma}: průměrný počet oprav {np.mean(updates):.0f}, "
          f"maximum {max(updates)}, mez {np.mean(bounds):.0f}")


controls(section2, gamma=choice("okraj γ", [0.05, 0.1, 0.2, 0.5, 1.0], 0.2))

**✏️ Úkol 2**

* Jak daleko je skutečný počet oprav od meze? Byla mez někdy překročena?
* Půlte γ (0,2 → 0,1 → 0,05). Kolikrát vzroste mez? A průměrný počet oprav?
* Je přímka perceptronu stejná jako pravá přímka x₁ + x₂ = 0? Proč ne?

## 3. Lineárně neoddělitelná data a kapsový algoritmus

**Cíl metody:** získat použitelnou přímku i tehdy, když žádná přímka třídy neoddělí. Perceptronové pravidlo se na takových datech
nikdy nezastaví: každá epocha obsahuje chyby a každá chyba přímku posune. **Kapsový algoritmus** si odkládá
(„do kapsy“) váhy s dosud nejmenším počtem trénovacích chyb a na konci vrátí je.

Dva mraky bodů se překrývají; překryv určuje vzdálenost jejich středů. Levý graf porovnává poslední váhy
s váhami v kapse, pravý ukazuje počet trénovacích chyb po každé opravě.

**Ovládání:** vzdálenost středů mraků (malá = velký překryv), počet epoch.

In [ ]:
def section3(distance, epochs):
    X, y = two_clouds(rng(13), 60, distance)
    g = rng(14)
    w, b = np.zeros(2), 0.0
    best_w, best_b, best_errors = w.copy(), b, count_errors(X, y, w, b)
    current, pocket = [], []
    for epoch in range(epochs):
        for i in g.permutation(len(y)):
            if y[i] * (w @ X[i] + b) <= 0:             # běžná oprava perceptronu
                w, b = w + y[i] * X[i], b + y[i]
                e = count_errors(X, y, w, b)           # chyby nových vah na všech trénovacích datech
                if e < best_errors:                    # lepší než kapsa -> do kapsy
                    best_w, best_b, best_errors = w.copy(), b, e
                current.append(e)
                pocket.append(best_errors)
    fig, axs = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 1.5]})
    show_classes(axs[0], X, y, s=16)
    draw_line(axs[0], w, b, (-5, 5), color=GREY, lw=2, ls="--", label="poslední váhy")
    draw_line(axs[0], best_w, best_b, (-5, 5), color=TEAL, lw=2.5, label="váhy v kapse")
    axs[0].set(xlim=(-5, 5), ylim=(-5, 5), xlabel="x₁", ylabel="x₂")
    axs[0].set_aspect("equal")
    axs[0].legend(loc="lower right", fontsize=8)
    axs[1].plot(current, color=GREY, lw=1, label="aktuální váhy")
    axs[1].step(range(len(pocket)), pocket, where="post", color=BLUE, lw=2.5, label="váhy v kapse")
    axs[1].set(xlabel="počet oprav", ylabel="trénovací chyby")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"oprav: {len(current)}, chyby posledních vah: {count_errors(X, y, w, b)}, "
          f"chyby v kapse: {best_errors} z {len(y)}")


controls(section3,
         distance=fslider("vzdálenost středů", 2.5, 1.0, 6.0, 0.5, fmt=".1f"),
         epochs=islider("epochy", 20, 1, 100))

**✏️ Úkol 3**

* Ustálí se s přibývajícími epochami počet chyb aktuálních vah? Jsou poslední váhy ty nejlepší?
* Zvyšte počet epoch na 100. Zlepší se kapsa? Zlepší se poslední váhy?
* Nastavte vzdálenost 6. Co se stane s počtem oprav? Proč se samotné perceptronové pravidlo na překrývajících se datech nemá jak zastavit?

## 4. Škálování vstupů

**Cíl metody:** dát příznakům srovnatelná měřítka, aby každá oprava posunula všechny váhy srovnatelně.
Oprava perceptronu přičítá η·y·**x**: příznak s hodnotami v tisících mění svou váhu tisíckrát víc než příznak
v jednotkách, zatímco práh se mění jen o η. **Standardizace** od každého příznaku odečte průměr a vydělí ho směrodatnou odchylkou.

Stejná oddělitelná data se učí dvakrát: s druhým příznakem v původních jednotkách (rozptyl a posun) a po standardizaci.

**Ovládání:** kolikrát větší je rozptyl x₂ a posun přičtený k x₂.

In [ ]:
def section4(scale_x2, offset_x2):
    Z, y = two_clouds(rng(15), 50, 3.4, gap=0.3)                    # oddělitelná data jako v přednášce
    X_raw = np.c_[Z[:, 0], offset_x2 + scale_x2 * Z[:, 1]]          # x₂ ve „velkých“ jednotkách
    X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)        # standardizace
    curves = {}
    for name, Xq in (("původní jednotky", X_raw), ("po standardizaci", X_std)):
        g, w, b, errs = rng(16), np.zeros(2), 0.0, []
        for epoch in range(40):
            w, b, _, _ = perceptron(Xq, y, w0=w, b0=b, g=g, epochs=1)  # jedna epocha, pokračuje z předchozích vah
            errs.append(count_errors(Xq, y, w, b))
        curves[name] = errs
    fig, axs = plt.subplots(1, 2)
    show_classes(axs[0], X_raw, y, s=14)
    axs[0].set(xlabel="x₁", ylabel="x₂ v původních jednotkách")
    for (name, errs), color, marker in zip(curves.items(), (ORANGE, BLUE), ("^", "o")):
        axs[1].plot(range(1, 41), errs, color=color, marker=marker, ms=4, label=name)
    axs[1].set(xlabel="epocha", ylabel="trénovací chyby po epoše")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    for name, errs in curves.items():
        print(f"{name}: chyby v posledních 10 epochách {errs[-10:]}")


controls(section4,
         scale_x2=choice("rozptyl x₂ (krát)", [1, 10, 200, 1000], 200),
         offset_x2=menu("posun x₂", [("0", 0), ("1000", 1000)], 1000))

**✏️ Úkol 4**

* Která varianta dosáhne nuly trénovacích chyb a po kolika epochách? Co místo toho dělá neškálovaná varianta a proč?
* Nastavte posun 0 a ponechte rozptyl 200. Škodí víc posun, nebo rozptyl? Nápověda: jak velká je oprava prahu ve srovnání s x₂?
* Pro rozptyl 1 a posun 0 se obě křivky téměř shodují. Proč?

## 5. Lineární neuron: gradientní sestup a rychlost učení

**Cíl metody:** proložit spojitými cíli přímku ŷ = w·x + b minimalizací **střední kvadratické chyby**
E = (1/n)·Σ(y − ŷ)². Gradientní sestup opakuje krok (w, b) ← (w, b) − η·∇E. Příliš malý krok konverguje pomalu,
příliš velký přeskakuje minimum a diverguje. Pro kvadratickou chybu lze hranici spočítat:
sestup konverguje pro η < 1/λ_max, kde λ_max je největší vlastní číslo matice momentů [[průměr x², průměr x], [průměr x, 1]].

Data jsou 40 regresních bodů z přednášky (interaktivní snímek o rychlosti učení). Levý graf ukazuje vrstevnice
E v rovině (w, b) a cestu sestupu z bodu (−0,5; 5), pravý graf E po každém kroku.

**Ovládání:** rychlost učení η, počet kroků, jednotka x (stejná data s x v desetkrát větších číslech).

In [ ]:
g = rng(17)
x5 = g.uniform(0, 4, 40)                        # regresní data z přednášky
y5 = 1.5 * x5 + 1 + g.normal(0, 0.6, 40)


def section5(eta, steps, x_unit):
    x, y = x5 * x_unit, y5                      # stejná data v jiné jednotce
    M = np.array([[np.mean(x ** 2), np.mean(x)], [np.mean(x), 1.0]])  # matice momentů
    eta_max = 1 / np.linalg.eigvalsh(M).max()   # hranice stability rychlosti učení
    w_opt, b_opt = np.polyfit(x, y, 1)          # přesné minimum pro srovnání
    if x_unit == 1:
        assert round(eta_max, 2) == 0.18        # hodnota z přednášky
    mse = lambda w, b: np.mean((w * x + b - y) ** 2)
    path, errors = [(-0.5 / x_unit, 5.0)], []
    with np.errstate(over="ignore", invalid="ignore"):
        for step in range(steps):
            w, b = path[-1]
            errors.append(mse(w, b))
            r = w * x + b - y                   # rezidua s opačným znaménkem
            grad = np.array([2 * np.mean(r * x), 2 * np.mean(r)])  # gradient E
            path.append(tuple(np.array([w, b]) - eta * grad))
            if not np.isfinite(path[-1]).all() or abs(path[-1][1]) > 1e6:
                break                           # divergence: kreslení končí
    path = np.array(path)
    W, Bb = np.meshgrid(np.linspace(w_opt - 2.5 / x_unit, w_opt + 2.5 / x_unit, 200), np.linspace(b_opt - 5, b_opt + 5, 200))
    E = np.mean((W[..., None] * x + Bb[..., None] - y) ** 2, axis=-1)
    fig, axs = plt.subplots(1, 2)
    axs[0].contour(W, Bb, E, levels=np.quantile(E, [0.002, 0.01, 0.03, 0.07, 0.15, 0.3, 0.5, 0.75]), colors=GREY, linewidths=1)
    axs[0].plot(path[:, 0], path[:, 1], color=ORANGE if eta >= eta_max else BLUE, marker="o", ms=3, lw=1.5)
    axs[0].scatter([w_opt], [b_opt], marker="*", s=200, color="black", zorder=5)
    axs[0].set(xlim=(W.min(), W.max()), ylim=(Bb.min(), Bb.max()), xlabel="w", ylabel="b", title=f"η = {eta:g}, η_max ≈ {eta_max:.3g}")
    axs[1].semilogy(range(len(errors)), errors, color=BLUE, marker="o", ms=3)
    axs[1].axhline(mse(w_opt, b_opt), color=GREY, ls="--", label="minimum E")
    axs[1].set(xlabel="krok", ylabel="E (log. měřítko)")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"hranice η_max = 1/λ_max ≈ {eta_max:.4f}; E po posledním kroku {errors[-1]:.4g}, "
          f"minimum {mse(w_opt, b_opt):.4g}")


controls(section5,
         eta=fslider("rychlost učení η", 0.12, 0.005, 0.2, 0.005, fmt=".3f"),
         steps=islider("počet kroků", 40, 10, 200, 10),
         x_unit=menu("jednotka x", [("původní", 1), ("x desetkrát větší", 10)], 1))

**✏️ Úkol 5**

* Najděte nejmenší η, pro kterou sestup diverguje. Jak blízko je vypsané hranici?
* Zhruba mezi 0,09 a 0,18 cesta kličkuje, ale stále konverguje. Ve kterém směru kličkuje a proč právě v něm?
* Přepněte jednotku x na „desetkrát větší“ (stejná data v jiných jednotkách). Jak se změnila η_max? Co to znamená pro volbu η bez škálování?

## 6. Logistický neuron a logaritmická ztráta

**Cíl metody:** získat místo tvrdého rozhodnutí **pravděpodobnost** třídy 1. Logistický neuron převede skóre
sigmoidou, p = σ(s) = 1 / (1 + e^(−s)), a učí se minimalizací **logaritmické ztráty** L = −ln p_y, kde p_y je
pravděpodobnost, kterou model dává správné třídě. Gradient má stejný tvar jako delta pravidlo: (p − y)·**x**.

Model se učí dávkovým gradientním sestupem na dvou překrývajících se mracích (třídy 0 a 1). Levý graf ukazuje vrstevnice
pravděpodobnosti 0,1, 0,5 a 0,9, prostřední průměrnou ztrátu po každé epoše, pravý ztrátu každého vzorku podle jeho skóre.

**Ovládání:** rychlost učení, počet epoch (gradientních kroků přes všechna data), vzdálenost středů mraků (zhruba od 6 jsou třídy oddělitelné).

In [ ]:
def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def section6(eta, epochs, distance):
    X, y_pm = two_clouds(rng(22), 60, distance)
    y = (y_pm > 0).astype(int)                       # třídy 0 a 1
    w, b, losses = np.zeros(2), 0.0, []
    with np.errstate(over="ignore", divide="ignore"):
        for epoch in range(epochs):
            p = sigmoid(X @ w + b)                   # pravděpodobnost třídy 1
            p_correct = np.where(y == 1, p, 1 - p)   # pravděpodobnost správné třídy
            losses.append(np.mean(-np.log(np.clip(p_correct, 1e-12, 1))))
            w = w - eta * np.mean((p - y)[:, None] * X, axis=0)  # gradientní krok: (p − y)·x
            b = b - eta * np.mean(p - y)
        xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
        P = sigmoid(np.c_[xx.ravel(), yy.ravel()] @ w + b).reshape(xx.shape)
        s = X @ w + b
        loss_each = -np.log(np.clip(np.where(y == 1, sigmoid(s), 1 - sigmoid(s)), 1e-12, 1))
        fig, axs = plt.subplots(1, 3, figsize=(13, 4))
        cs = axs[0].contour(xx, yy, P, levels=[0.1, 0.5, 0.9], colors=[BLUE, "black", ORANGE])
        axs[0].clabel(cs, fontsize=9)
        show_classes(axs[0], X, y, s=14)
        axs[0].set(xlabel="x₁", ylabel="x₂", xlim=(-5, 5), ylim=(-5, 5))
        axs[0].set_aspect("equal")
        axs[1].plot(losses, color=BLUE)
        axs[1].set(xlabel="epocha", ylabel="průměrná log. ztráta", ylim=(0, None))
        grid_s = np.linspace(min(-6, s.min()), max(6, s.max()), 300)
        axs[2].plot(grid_s, -np.log(sigmoid(grid_s)), color=ORANGE, label="y = 1: −ln σ(s)")
        axs[2].plot(grid_s, -np.log(1 - sigmoid(grid_s)), color=BLUE, label="y = 0: −ln(1 − σ(s))")
        axs[2].scatter(s, loss_each, c=np.where(y == 1, ORANGE, BLUE), s=10, zorder=3)
        axs[2].set(xlabel="skóre s", ylabel="ztráta vzorku", ylim=(0, 6))
        axs[2].legend(fontsize=8)
        plt.tight_layout()
        plt.show()
    print(f"w = ({w[0]:.2f}, {w[1]:.2f}), b = {b:.2f}, ‖w‖ = {np.linalg.norm(w):.2f}, výsledná průměrná ztráta {losses[-1]:.3f}")
    print("ztráta pro p_y = 0,9; 0,5; 0,01:", [round(float(-np.log(q)), 2) for q in (0.9, 0.5, 0.01)])


controls(section6,
         eta=choice("rychlost učení", [0.01, 0.1, 0.5, 2.0, 5.0, 20.0], 0.5),
         epochs=choice("epochy", [10, 50, 200, 1000, 2000, 5000], 200),
         distance=fslider("vzdálenost středů", 2.5, 1.0, 8.0, 0.5, fmt=".1f"))

**✏️ Úkol 6**

* Které vzorky přispívají ke ztrátě nejvíc? Jaká je ztráta vzorku, kterému model dává jen 1 % pro jeho správnou třídu?
* Nastavte rychlost učení 20. Klesá ztráta stále? Srovnejte s částí 5: proč zde nedochází k prudké divergenci?
* Nastavte vzdálenost 8 (oddělitelné třídy) a zvyšte počet epoch na 2000 a 5000. Co se stane s ‖w‖ a se šířkou pásu mezi vrstevnicemi 0,1 a 0,9? Proč to volá po regularizaci vah?

## 7. Matice záměn a rozhodovací práh

**Cíl metody:** popsat, *jaké* chyby klasifikátor dělá. **Matice záměn** počítá skutečně negativní (TN),
falešně pozitivní (FP), falešně negativní (FN) a skutečně pozitivní (TP). Z ní:

| Metrika | Vzorec | Na co odpovídá |
|---|---|---|
| správnost (accuracy) | (TP + TN) / n | kolik předpovědí je správných |
| přesnost (precision) | TP / (TP + FP) | kolik poplachů je skutečných |
| úplnost (recall, senzitivita) | TP / (TP + FN) | kolik pozitivních se najde |
| F1 | 2·P·R / (P + R) | harmonický průměr přesnosti a úplnosti |
| specificita | TN / (TN + FP) | kolik negativních zůstane v klidu |
| vyvážená správnost | (úplnost + specificita) / 2 | správnost nezávislá na podílu tříd |

Model dává skóre; **práh** z něj dělá rozhodnutí. Skóre negativních mají rozdělení N(0, 1), pozitivních
N(2, 1), jako na interaktivním snímku o prahu v přednášce. První řádek kontroluje funkci metrik na příkladu z přednášky
(TN 940, FP 10, FN 20, TP 30).

**Ovládání:** podíl pozitivních, práh (graf sleduje posuvník už během tažení).

In [ ]:
ACC, PREC, REC, SPEC, BAL = "správnost", "přesnost", "úplnost", "specificita", "vyvážená správnost"


def metrics(tn, fp, fn, tp):
    # všechny metriky ze čtyř počtů matice záměn
    precision = tp / (tp + fp) if tp + fp else float("nan")   # nedefinovaná, když nejsou poplachy
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {ACC: (tp + tn) / (tn + fp + fn + tp), PREC: precision, REC: recall,
            "F1": f1, SPEC: specificity, BAL: (recall + specificity) / 2}


lecture = metrics(940, 10, 20, 30)
assert round(lecture[ACC], 3) == 0.97 and round(lecture["F1"], 3) == 0.667 and round(lecture[BAL], 3) == 0.795
print("příklad z přednášky:", {k: round(v, 3) for k, v in lecture.items()})


def section7(prevalence, threshold):
    g = rng(26)
    n_pos = int(1000 * prevalence)
    scores = np.r_[g.normal(2, 1, n_pos), g.normal(0, 1, 1000 - n_pos)]
    y = np.r_[np.ones(n_pos), np.zeros(1000 - n_pos)].astype(int)

    def confusion(t):
        pred = scores >= t
        return (int(np.sum(~pred & (y == 0))), int(np.sum(pred & (y == 0))),
                int(np.sum(~pred & (y == 1))), int(np.sum(pred & (y == 1))))

    tn, fp, fn, tp = confusion(threshold)
    table = pd.DataFrame([metrics(tn, fp, fn, tp), metrics(1000 - n_pos, 0, n_pos, 0)],
                         index=[f"práh {threshold:.2f}", "vždy 0"]).round(3)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [1.4, 0.8, 1.4]})
    bins = np.linspace(-4, 6, 41)
    axs[0].hist(scores[y == 0], bins=bins, color=BLUE, alpha=0.6, label="negativní (y = 0)")
    axs[0].hist(scores[y == 1], bins=bins, color=ORANGE, alpha=0.6, label="pozitivní (y = 1)")
    axs[0].axvline(threshold, color="black", lw=2)
    axs[0].set(xlabel="skóre", ylabel="počet")
    axs[0].legend(fontsize=8)
    axs[1].imshow([[tn, fp], [fn, tp]], cmap="Blues")
    for (r, c), v in np.ndenumerate(np.array([[tn, fp], [fn, tp]])):
        axs[1].text(c, r, f"{['TN', 'FP', 'FN', 'TP'][2 * r + c]}\n{v}", ha="center", va="center", fontsize=12)
    axs[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["ŷ = 0", "ŷ = 1"], yticklabels=["y = 0", "y = 1"])
    axs[1].grid(False)
    ths = np.linspace(-2, 4, 121)
    curves = pd.DataFrame([metrics(*confusion(t)) for t in ths], index=ths)
    for name, color in ((ACC, GREY), (PREC, BLUE), (REC, ORANGE), ("F1", TEAL)):
        axs[2].plot(ths, curves[name], color=color, label=name)
    axs[2].axvline(threshold, color="black", lw=1)
    axs[2].set(xlabel="práh", ylim=(0, 1.02))
    axs[2].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    display(table)
    print(f"nejvyšší F1 při prahu {curves['F1'].idxmax():.2f}, nejvyšší správnost při prahu {curves[ACC].idxmax():.2f}")


controls(section7,
         prevalence=fslider("podíl pozitivních", 0.2, 0.01, 0.5, 0.01),
         threshold=fslider("práh", 1.0, -2.0, 4.0, 0.05, live=True))

**✏️ Úkol 7**

* Nastavte podíl pozitivních 0,05. Porovnejte správnost modelu a modelu „vždy 0“. Které metriky prozradí, že „vždy 0“ je k ničemu?
* Posuňte práh z 2 na 0. Jak se mění přesnost a úplnost? Proč se musí pohybovat opačně?
* Shodují se prahy s nejvyšším F1 a s nejvyšší správností? Který byste použili, je-li přehlédnutý pozitivní případ desetkrát horší než planý poplach?

## 8. ROC, přesnost–úplnost a cena chyb

**Cíl metod:** ohodnotit model přes **všechny** prahy a potom zvolit jeden práh podle důsledků chyb.

- **Křivka ROC** vynáší úplnost (TPR) proti podílu planých poplachů FPR = 1 − specificita. **AUC** je plocha pod ní:
  pravděpodobnost, že náhodný pozitivní vzorek dostane vyšší skóre než náhodný negativní. Pro dvě normální rozdělení ve vzdálenosti μ
  je rovna Φ(μ/√2).
- **Křivka přesnost–úplnost** vynáší přesnost proti úplnosti; její základní úrovní je podíl pozitivních.
- **Cena chyb:** při ceně c_FN za přehlédnutý pozitivní případ a c_FP za planý poplach je pro **kalibrovanou**
  pravděpodobnost p nejlevnější rozhodnutí „pozitivní, když p ≥ c_FP / (c_FP + c_FN)“.

Pravděpodobnosti jsou zde přesné (spočtené ze známých rozdělení), takže jsou kalibrované z definice.

**Ovládání:** vzdálenost μ středních skóre tříd, podíl pozitivních, cena přehlédnutého pozitivního případu c_FN, cena planého poplachu c_FP.

In [ ]:
def section8(distance, prevalence, c_fn, c_fp):
    g = rng(27)
    n = 5000
    n_pos = int(n * prevalence)
    scores = np.r_[g.normal(distance, 1, n_pos), g.normal(0, 1, n - n_pos)]
    y = np.r_[np.ones(n_pos), np.zeros(n - n_pos)].astype(int)
    like_pos = prevalence * norm.pdf(scores, distance, 1)           # Bayesův vzorec se známými rozděleními
    p = like_pos / (like_pos + (1 - prevalence) * norm.pdf(scores, 0, 1))  # přesná pravděpodobnost třídy 1
    order = np.argsort(-scores)                     # prahy od nejvyššího skóre dolů
    tp = np.cumsum(y[order] == 1)
    fp = np.cumsum(y[order] == 0)
    tpr, fpr = np.r_[0, tp / n_pos], np.r_[0, fp / (n - n_pos)]
    precision = tp / (tp + fp)
    auc = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)  # plocha pod křivkou ROC
    cuts = np.round(np.arange(0.01, 1.0, 0.01), 2)  # prahy na pravděpodobnosti
    cost = np.array([c_fn * np.sum((p < t) & (y == 1)) + c_fp * np.sum((p >= t) & (y == 0)) for t in cuts])
    best = cuts[np.argmin(cost)]
    theory = c_fp / (c_fp + c_fn)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].plot(fpr, tpr, color=BLUE, lw=2)
    axs[0].plot([0, 1], [0, 1], color=GREY, ls="--")
    axs[0].set(xlabel="FPR = 1 − specificita", ylabel="úplnost (TPR)", title=f"ROC, AUC = {auc:.3f}")
    axs[1].plot(tp / n_pos, precision, color=ORANGE, lw=2)
    axs[1].axhline(prevalence, color=GREY, ls="--", label="podíl pozitivních")
    axs[1].set(xlabel="úplnost", ylabel="přesnost", ylim=(0, 1.02), title="přesnost–úplnost")
    axs[1].legend(fontsize=8)
    axs[2].plot(cuts, cost, color=TEAL, lw=2)
    axs[2].axvline(theory, color=ORANGE, ls="--", label=f"c_FP/(c_FP + c_FN) = {theory:.3f}")
    axs[2].scatter([best], [cost.min()], color="black", zorder=3, label=f"minimum při {best:.2f}")
    axs[2].set(xlabel="práh na pravděpodobnosti p", ylabel="celková cena")
    axs[2].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"AUC {auc:.3f}, teorie Φ(μ/√2) = {norm.cdf(distance / np.sqrt(2)):.3f}; "
          f"nejlevnější práh {best:.2f} (teorie {theory:.3f}), "
          f"cena {cost.min()} vs. práh 0,5: {cost[cuts == 0.5][0]}, „vždy 0“: {c_fn * n_pos}")


controls(section8,
         distance=fslider("vzdálenost μ", 2.0, 0.0, 4.0, 0.25),
         prevalence=fslider("podíl pozitivních", 0.2, 0.01, 0.5, 0.01),
         c_fn=islider("cena přehlédnutí c_FN", 10, 1, 20),
         c_fp=islider("cena planého poplachu c_FP", 1, 1, 20))

**✏️ Úkol 8**

* Změňte podíl pozitivních z 0,2 na 0,02. Změní se křivka ROC nebo AUC? A křivka přesnost–úplnost? Proč ten rozdíl?
* Jak blízko je nejlevnější práh hodnotě c_FP / (c_FP + c_FN)? Nastavte c_FN = 1: kde je optimum teď?
* Proč vzorec platí jen pro kalibrované pravděpodobnosti? Co by se stalo se surovým skóre nebo s modelem učeným na převážených třídách?

## 9. Metriky regrese a odlehlá hodnota

**Cíl metod:** shrnout rezidua e = y − ŷ regresního modelu do jednoho čísla.

| Metrika | Vzorec | Vlastnost |
|---|---|---|
| MAE | (1/n)·Σ\|e\| | v jednotkách cíle, robustní |
| MSE | (1/n)·Σe² | čtverce: velké chyby převáží |
| RMSE | √MSE | v jednotkách cíle, RMSE ≥ MAE |
| R² | 1 − Σe² / Σ(y − ȳ)² | podíl vysvětleného rozptylu; 0 = stejně dobré jako průměr, může být záporné |
| MAPE | (100 %/n)·Σ\|e / y\| | relativní; nestabilní pro cíle blízko nuly |

Data jsou deset vzorků z přednášky (cíle 10 až 28, rezidua ±1 a ±2). První řádek tabulky je
původní model, druhý vaše nastavení.

**Ovládání:** reziduum posledního vzorku (původně −1; odlehlá hodnota z přednášky je 12), posun odečtený od všech cílů i předpovědí.

In [ ]:
y9 = np.arange(10, 29, 2, dtype=float)                      # cíle 10, 12, …, 28
e9 = np.array([1, -1, 2, -2, 1, -1, 2, -2, 1, -1], dtype=float)  # rezidua z přednášky


def regression_metrics(y, y_hat):
    e = y - y_hat
    return {"MAE": np.mean(np.abs(e)), "MSE": np.mean(e ** 2), "RMSE": np.sqrt(np.mean(e ** 2)),
            "R²": 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2), "MAPE %": 100 * np.mean(np.abs(e / y))}


def section9(last_residual, shift):
    e_new = e9.copy()
    e_new[-1] = last_residual
    y_new = y9 - shift
    y_hat_new = y_new - e_new                                  # předpovědi se změněným reziduem
    rows = {"původní": regression_metrics(y9, y9 - e9), "vaše nastavení": regression_metrics(y_new, y_hat_new)}
    table = pd.DataFrame(rows).T.round(3)
    assert tuple(table.iloc[0][["MAE", "MSE", "RMSE", "R²"]]) == (1.4, 2.2, 1.483, 0.933)  # hodnoty z přednášky
    fig, axs = plt.subplots(1, 2)
    lim = (min(y_new.min(), y_hat_new.min()) - 2, max(y_new.max(), y_hat_new.max()) + 2)
    axs[0].plot(lim, lim, color=GREY, ls="--")
    axs[0].scatter(y_new, y_hat_new, color=TEAL, s=50)
    axs[0].set(xlim=lim, ylim=lim, xlabel="skutečná hodnota y", ylabel="předpověď ŷ")
    axs[0].set_aspect("equal")
    axs[1].bar(range(1, 11), e_new, color=[ORANGE if abs(v) > 3 else BLUE for v in e_new])
    axs[1].axhline(0, color="black", lw=1)
    axs[1].set(xlabel="vzorek", ylabel="reziduum e = y − ŷ", xticks=range(1, 11))
    plt.tight_layout()
    plt.show()
    display(table)


controls(section9,
         last_residual=islider("reziduum posledního vzorku", 12, -5, 25, live=True),
         shift=fslider("posun cílů", 0.0, 0.0, 9.5, 0.5, fmt=".1f", live=True))

**✏️ Úkol 9**

* Která metrika s odlehlou hodnotou 12 relativně vzroste nejvíc a proč? Kterou byste uvedli někomu, kdo chce „typickou chybu v jednotkách“?
* Najděte reziduum, při kterém je R² záporné. Co záporné R² znamená?
* Nastavte reziduum −1 a posun 9,5 (cíle blízko nuly). Které metriky zůstanou stejné a která vybuchne? Proč?

## 10. XOR a nový příznak

**Cíl metody:** rozpoznat úlohu, kterou jeden neuron nevyřeší, a jednu cestu, jak ji obejít. V XOR jsou body (0, 1) a (1, 0)
ve třídě +1, body (0, 0) a (1, 1) ve třídě −1. Žádná přímka je neoddělí, perceptron proto nikdy nedokončí epochu bez
opravy. Přidání **součinu x₁·x₂** jako třetího příznaku udělá třídy oddělitelnými ve třech rozměrech; v původní rovině
je hranicí křivka. Druhá cesta – více neuronů ve vrstvách – je tématem přednášky o vícevrstvých sítích.

**Ovládání:** přidání příznaku x₁·x₂, počet epoch.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])


def section10(add_product, epochs):
    def features(X):
        # původní příznaky, případně s jejich součinem
        return np.c_[X, X[:, 0] * X[:, 1]] if add_product else X

    F = features(X_xor)
    w, b, errs, hist = perceptron(F, y_xor, w0=np.zeros(F.shape[1]), epochs=epochs)
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
    S = (features(np.c_[xx.ravel(), yy.ravel()]) @ w + b).reshape(xx.shape)
    fig, axs = plt.subplots(1, 2)
    axs[0].contourf(xx, yy, S, levels=[-1e9, 0, 1e9], colors=[BLUE, ORANGE], alpha=0.15)
    if S.min() < 0 < S.max():
        axs[0].contour(xx, yy, S, levels=[0], colors="black")
    show_classes(axs[0], X_xor, y_xor, s=120)
    axs[0].set(xlabel="x₁", ylabel="x₂", xticks=[0, 1], yticks=[0, 1],
               title=f"trénovací chyby: {count_errors(F, y_xor, w, b)}")
    axs[0].set_aspect("equal")
    axs[1].plot(range(1, len(errs) + 1), errs, color=ORANGE, marker="o")
    axs[1].set(xlabel="epocha", ylabel="opravy v epoše", ylim=(0, 4.5),
               xticks=range(1, len(errs) + 1, max(1, len(errs) // 10)))
    plt.tight_layout()
    plt.show()
    print("w =", np.round(w, 2), " b =", b, " opravy po epochách:", errs)


controls(section10,
         add_product=widgets.Checkbox(value=False, description="přidat příznak x₁·x₂", style=STYLE),
         epochs=islider("epochy", 20, 1, 50))

**✏️ Úkol 10**

* Proč počet oprav bez součinu nikdy neklesne na nulu? Pomohlo by jiné pořadí nebo rychlost učení?
* Zaškrtněte příznak x₁·x₂. Po kolika epochách se perceptron zastaví? Jaké znaménko má váha x₁·x₂ a proč?
* Jak vypadá hranice v původní rovině? Proč lze model se součinem stále nazvat „lineárním“?